### Paso 3: Limpiar Datos

Este archivo va a seguir movimientos similares al del autor original


In [111]:
#### Importacion de Datos
import pandas as pd

In [112]:
#### Leyendo los datos 
    # OJO: todos los valores estan en string
df = pd.read_csv('glassdoor_jobs.csv')
df = df.drop('Unnamed: 0', axis=1)



In [113]:
df.columns

Index(['Job Title', 'Salary Estimate', 'Job Description', 'Rating',
       'Company Name', 'Location', 'Headquarters', 'Size', 'Founded',
       'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors'],
      dtype='str')

#### Arreglando Variable Dependiente

In [114]:
#### Investigando la varaible dependiente
    # La variable dependiente, basado en Formulacion del Problema, es
            # el sueldo de una posicon de DS
    # Por ende, la variable dependiente ya se encuentra en este DF
        # Varaible: "Salary Estimate"
        # PERO tiene mucho fluff adicionale
    # En este bloque vamos a limpiar la varaible dependiente
VD = 'Salary Estimate'
df[VD].value_counts()
    # Existen varios registros con valor "-1"
    # Existen registros con "Per Hour" o "Employer provided salary"
    # TODOS tienen "(Glassdoor...)"

#### Creando columna para sueldos "per hour" o "employer provided salary"
    # Estos significan que los valores sin FIJOS y no ESTIMADOS
### Creando columna para "hora"
df['cada_hora'] = df[VD].apply(lambda x: 1 if 'hour' in x.lower() else 0)
### Creando columna para "Epxloyer provided"
df['empleador'] = df[VD].apply(lambda x: 1 if 'employer' in x.lower() else 0)


In [115]:
#### Eliminando sueldos que tengan -1
df = df[df[VD] != '-1']

In [116]:
#### Limpiando el formato de sueldos
sueldo = df[VD].apply(lambda x: x.split('(')[0])
    # Eliminando "(Glassdoor...)"
sueldo_limpio = sueldo.apply(lambda x:
    x.lower()
    .replace('k', '')   # Elimiando 'k'
    .replace('$', '')   # Elimando '$' 
    .replace('per hour', '')    # Elimanando 'per hour'
    .replace('employer provided salary:', '')
)


#### Creando nueva columnas de sueldo
df['sueldo_min'] = sueldo_limpio.apply(lambda x: int(x.split('-')[0]))
df['sueldo_max'] = sueldo_limpio.apply(lambda x: int(x.split('-')[1]))
df['sueldo_promedio'] = (df['sueldo_min'] + df['sueldo_max'])/2

In [117]:
#### Arreglando sueldo de hourly vs annual
    # necesita considerar dos cosas:
        # Se elimino la "K" de 1000
        # Existen apprx 2000 horas en un ano, para considerar con hourly sueldo
        # x $/hora * 2000 horas/ ano * unit_final/1000(K) = x * 2
df['sueldo_min'] = df.apply(lambda x: x['sueldo_min']*2 if x['cada_hora'] else x['sueldo_min'], axis=1)
df['sueldo_max'] = df.apply(lambda x: x['sueldo_max']*2 if x['cada_hora'] else x['sueldo_max'], axis=1)
df['sueldo_promedio'] = df.apply(lambda x: x['sueldo_promedio']*2 if x['cada_hora'] else x['sueldo_promedio'], axis=1)

In [118]:
##### Arreglando Variables independientes
    # Ahora falta arregalar todos los variables indeendientes
    # Lista: Job Title, Job Description, Rating,  etc.
df.columns


Index(['Job Title', 'Salary Estimate', 'Job Description', 'Rating',
       'Company Name', 'Location', 'Headquarters', 'Size', 'Founded',
       'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors',
       'cada_hora', 'empleador', 'sueldo_min', 'sueldo_max',
       'sueldo_promedio'],
      dtype='str')

#### Arreglando VI: Titulo

In [119]:
#### Arreglado Job Title
### Creando titulos de trabajos
def simplificando_titulo(titulo):
    if 'data scientist' in titulo.lower(): return 'cientifico de datos'
    elif 'data engineer' in titulo.lower(): return 'ingeniero de datos'
    elif 'analyst' in titulo.lower(): return 'analysta de datos'
    elif 'machine learning' in titulo.lower(): return 'ingeniero de ML'
    elif 'consultant' in titulo.lower(): return 'asesor'
    elif 'manager' in titulo.lower(): return 'gerente'
    elif 'director' in titulo.lower(): return 'director'
    elif 'principal' in titulo.lower(): return  'principal'
    else: return 'na'

### Creando senioridad de trabajo
def senioridad(titulo):
    if 'sr' in titulo.lower() or 'senior' in titulo.lower():
        return 'senior'
    elif 'jr' in titulo.lower() or 'junior' in titulo.lower():
        return 'junior'
    else: return 'na'

df['titulo'] = df['Job Title'].apply(simplificando_titulo)
df['senioridad'] = df['Job Title'].apply(senioridad)


In [120]:
#### Anlizando arregles de titulos
df['titulo'].value_counts()
# df['senioridad'].value_counts()

titulo
cientifico de datos    279
na                     162
ingeniero de datos     119
analysta de datos      102
gerente                 22
ingeniero de ML         22
director                14
principal               14
asesor                   8
Name: count, dtype: int64

#### Arreglando VI: Descripcion

In [121]:
#### Investigando las desripciones
### Python
df['python_yn'] = df['Job Description'].apply(lambda x: 1 if 'python' in x.lower() else 0)

### R studio
df['r_yn'] = df['Job Description'].apply(lambda x: 1 if 'r studio' in x.lower() else 0)

### Spark 
df['spark_yn'] = df['Job Description'].apply(lambda x: 1 if 'spark' in x.lower() else 0)

### AWS
df['aws_yn'] = df['Job Description'].apply(lambda x: 1 if 'aws' in x.lower() else 0)

### Excel
df['excel_yn'] = df['Job Description'].apply(lambda x: 1 if 'excel' in x.lower() else 0)

In [122]:
##### Viendo el tamano de la descripcion
    # Estos valores se utilizaran para ver si el tamano de la descripcion tiene una relacion con la VD: Sueldo
df['len_desc'] = df['Job Description'].apply(lambda x: len(x))

### Arreglando VI: Rating y Nombre de Compania

In [123]:
##### Analises
    # Categoria 'Rating' esta bien
    # Pero el 'Company Name' depende en el rating
        # si 'Rating' >-1: El 'Rating' esta integrando dentro del nombre 
        # si 'Rating' == -1: El 'Rating' no se encuentra dentro el nombre mismo

df['compania'] = df.apply(lambda x: x['Company Name'] if x['Rating'] < 0 else x['Company Name'][:-4], axis=1)

### Arreglando VI: Estado

In [124]:
##### Estados
    # Solo se va a extraer los estados, no las ciudades

df['estado'] = df['Location'].apply(lambda x: x.split(' ')[-1])
df['estado'].value_counts()

estado
CA    152
MA    103
NY     72
VA     41
IL     40
MD     35
PA     33
TX     28
WA     21
NC     21
NJ     17
FL     16
OH     14
TN     13
CO     11
DC     11
IN     10
WI     10
UT     10
MO      9
AZ      9
AL      8
KY      6
MI      6
GA      6
DE      6
CT      5
IA      5
OR      4
LA      4
NE      4
NM      3
KS      3
MN      2
ID      2
RI      1
SC      1
Name: count, dtype: int64

### Arreglando VI: HQ

In [125]:
#### HQ
    # Ver si el HQ esta dentro la ubicacion del trabajo

df['HQ'] = df.apply(lambda x: 1 if x['Headquarters'] == x['Location'] else 0, axis =1)


### Arreglando VI: Size

In [126]:
#### Borrando 'employees' y 'to'-> '-' de size
df['tamano'] = df['Size'].apply(lambda x: x.replace('employees', '').replace(' to ', '-'))

### Arreglando VI: Founded

In [127]:
#### convirtiendo "Founded" en anos desde el ano cuando se extrayo este DF
df['ano'] = df['Founded'].apply(lambda x: x if x < 1 else 2020 - x)

### Arreglando VI: ownership 

In [128]:
##### Cambiando las categorias
def tipo_org(org: str):
    if 'private' in org.lower(): return 'privado'
    elif 'public' in org.lower(): return 'publico'
    elif 'nonprofit' in org.lower(): return 'sin_lucro'
    elif 'government' in org.lower(): return 'gubernamental'
    elif 'subsidiary' in org.lower(): return 'subsidiary'
    elif 'hospital' in org.lower(): return 'hospital'
    elif 'university' in org.lower(): return 'universidad'
    elif 'school' in org.lower(): return 'colegio'
    else: return 'na'

In [129]:
df['org'] = df['Type of ownership'].apply(tipo_org)
df['org'].value_counts()

org
privado          410
publico          193
sin_lucro         55
subsidiary        34
gubernamental     15
hospital          15
universidad       13
na                 5
colegio            2
Name: count, dtype: int64

#### VIs: Industry, Sector, Revenue - No se van a tocar
Ya estan en un formato consistente

### Arreglando VI: Competitors

In [130]:
#### Avergiuando cuando comeptidores tiene cada listing/compania
df['num_comps'] = df['Competitors'].apply(lambda x: int(x) if x=='-1' else len(x.split(',')))
df['num_comps'].value_counts()

num_comps
-1    460
 3    228
 2     41
 1     12
 4      1
Name: count, dtype: int64

### Borrando Columnas no necesarias o que ya estan limpias

Columnas: 'Job Title', 'Salary Estimate', 'Founded', 'Job Description', 'Company Name', 'Location', 'Headquarters', 'Size', 'Type of ownership', 'Competitors'

In [131]:
#### Elimnando columnas
df_limpio = df.drop(columns=['Job Title', 'Salary Estimate', 'Founded', 'Job Description', 'Company Name', 'Location', 'Headquarters', 'Size', 'Type of ownership', 'Competitors'])

In [135]:
#### Guardando DF limipio
df_limpio.to_csv('datos_limpios.csv', index=False)